Reading the Dataset
We will read the dataset and csv file using pandas also we will use some more libraries like numpy and matplotlib for data manipulation and visualization.

In [1]:
import pandas as pd  # For data manipulation
import numpy as np # For numerical operations
import time
import random  # For random number generation
from typing import List, Tuple # For type hinting

In [4]:
# Loading the dataset
cities_df = pd.read_csv('../data/cities.csv')
cities_df.head()

,City,X,Y
0,City_1,37.454012,95.071431
1,City_2,73.199394,59.865848
2,City_3,15.601864,15.599452
3,City_4,5.808361,86.617615
4,City_5,60.111501,70.807258


# Here we will select some random 5 cities as a part of sampling

In [5]:
# Selecting the 5 samples from dataset
sample_cities = cities_df.head(5)
sample_cities

,City,X,Y
0,City_1,37.454012,95.071431
1,City_2,73.199394,59.865848
2,City_3,15.601864,15.599452
3,City_4,5.808361,86.617615
4,City_5,60.111501,70.807258


# Measuring distance : Now we will follow euclidean distance to find the shortest distance between 2 points.

# Reference : https://en.wikipedia.org/wiki/Euclidean_distance , https://flexiple.com/python/calculate-distance-python

In [6]:
#Function for calculating euclidean distance
def euclidean_distance(city1: Tuple[float, float], city2: Tuple[float, float]) -> float:
    return np.sqrt((city1[0] - city2[0])**2 + (city1[1] - city2[1])**2)

In [7]:
#Sample distance calculation between 2 cities
"""
This is a sample how the line of code is calculating distance . 
From the sample_cities variable we are taking the iloc and the X and Y coordinate values. After that passing this in the euclidean 
distance function which is calculating the distance between 2 points
"""
print(euclidean_distance((sample_cities.iloc[0]['X'], sample_cities.iloc[0]['Y']), (sample_cities.iloc[1]['X'], sample_cities.iloc[1]['Y'])))

50.17136010935256


Now we will create a function which will calculate total distance from a list

In [12]:
# Here this function does a total tour and calculates the net covered distance
# Each time the loop is running it is adding distance covered and upto returning
"""
This function is calculating total distance covered. Here we are passing a list of the available paths and then it is going in 
a loop which is adding distance between 2 cities and then doing a sum
"""
def calculate_total_distance(tour: List[int], cities: np.ndarray) -> float:
    total_dist = 0.0
    for i in range(len(tour)):
        city1 = cities[tour[i]]
        city2 = cities[tour[(i + 1) % len(tour)]]
        total_dist += euclidean_distance(city1, city2)
    return total_dist

In [13]:
#Here I am extracting the coordinates
cities = sample_cities[['X', 'Y']].values.astype(float)
cities

array([[37.45401188, 95.07143064],
       [73.19939418, 59.86584842],
       [15.60186404, 15.59945203],
       [ 5.80836122, 86.61761458],
       [60.11150117, 70.80725778]])

In [14]:
#Example usage
cities = sample_cities[['X', 'Y']].values.astype(float)
calculate_total_distance([0, 1, 2, 3, 4], cities)
#

np.float64(284.26049177638447)

# Part 2 : Single Solution Algorithm Implementation

For better understanding and clean code we will apply the theory of object orientation. We will create a class for each individual type of algorithm

Reference : https://machinelearningmastery.com/simulated-annealing-from-scratch-in-python/

In [25]:
class SimulatedAnnealing:
    """
    Simulated Annealing algorithm for solving TSP.

    This algorithm uses a probabilistic approach to escape local optima
    by accepting worse solutions with a probability that decreases over time.

    Time Complexity: O(iterations * n) where n is number of cities
    Space Complexity: O(n) for storing the tour
    """

    def __init__(self, cities: np.ndarray, initial_temp: float = 10000,
                 cooling_rate: float = 0.995, min_temp: float = 0.1):
        """
        Initialize Simulated Annealing algorithm.
        
        Args:
            cities: Array of city coordinates
            initial_temp: Starting temperature
            cooling_rate: Rate at which temperature decreases
            min_temp: Minimum temperature threshold
        """
        self.cities = cities
        self.n_cities = len(cities)
        self.initial_temp = initial_temp
        self.cooling_rate = cooling_rate
        self.min_temp = min_temp
        self.best_tour = None
        self.best_distance = float('inf')
        self.history = []

    def get_initial_solution(self):
        """
        Generate initial random solution.

        Time Complexity: O(n)
        Space Complexity: O(n)
        """
        tour = list(range(self.n_cities))
        random.shuffle(tour)
        return tour

    def get_neighbor(self, tour: List[int]):
        """
        Generating neighbor solution using 2-opt swap.

        Time Complexity: O(n)
        Space Complexity: O(n)
        """
        new_tour = tour.copy()
        i, j = sorted(random.sample(range(self.n_cities), 2))
        new_tour[i:j+1] = reversed(new_tour[i:j+1])
        return new_tour   

    def acceptance_probability(self, current_dist: float, new_dist: float,
                              temp: float):
        """
        Here we will calculate the probability of acccepting a worse solution 
        
        Time Complexity: O(1)
        Space Complexity: O(1)
        """
        if new_dist < current_dist:
            return 1.0
        return np.exp((current_dist - new_dist) / temp) 

    def solve(self, max_iterations: int = 10000):
        """
        here max_iterations is the Maximum number of iterations to run, we need a number of iterations for this algorithm
    
        This is how the function will work. 
        It will start with a random tour, then will try a small random change in the neighbor. If the tour is better it will 
        accept it else if it is worse then it will accept with a probability which depends on the temperature. 

        """
    
        # Generating initial random solution at first 
        current_tour = self.get_initial_solution()
    
        # After getting the current tour we will calculate the total distance which we declared earlier
        current_dist = calculate_total_distance(current_tour, self.cities)
    
        # Store the current best solution
        self.best_tour = current_tour.copy()
        self.best_distance = current_dist
    

        # Here we will initialize simulated annealing temperature and iteration counter
        temperature = self.initial_temp
        iteration = 0
    

        # In this part we call a loop this loop is under while loop and it will run until the temperature is too low
        # Also after reaching maximum iterations
        while temperature > self.min_temp and iteration < max_iterations:
    
            # Here we are generating a neighbor of the current solution 
            new_tour = self.get_neighbor(current_tour)
    
            # Now calculating the total distance of the tour using previously declared function
            new_dist = calculate_total_distance(new_tour, self.cities)

            """Here now we will decide whether to accept the new tour or reject
            We compare new distance with current distance if the current distance is smaller than new distance then we 
            accept it other wise we will accept with probability 
            """

            if self.acceptance_probability(current_dist, new_dist, temperature) > random.random():
                # Accept move
                current_tour = new_tour
                current_dist = new_dist

                if current_dist < self.best_distance:
                    self.best_tour = current_tour.copy()
                    self.best_distance = current_dist
    
            # Here we will cool down the temperature by updating the old cooling temperature with cooling rate
            # This reduces the probability of accepting worse moves
            # as the algorithm progresses.
            temperature *= self.cooling_rate
    
            # Increasing iteration counter
            iteration += 1
    
            # Saving every 100 iterations
            if iteration % 100 == 0:
                self.history.append({
                    'iteration': iteration,
                    'temperature': temperature,
                    'current_distance': current_dist,
                    'best_distance': self.best_distance
                })
    
        return self.best_tour, self.best_distance
